In [24]:
import torch
import torch.nn as nn
from pprint import pprint
from patch_classify_loss import PatchClassifyBCELoss


### Testing patch classify BCE loss (with pos_wts)

In [1]:
def create_patch_classify_preds_and_target_binary(**kwargs):
    patch_sizes = kwargs.get('patch_sizes', [8, 16, 32])
    im_size = kwargs.get('im_size', 128)
    n_channels = 1
    batch_size = kwargs.get('batch_size', 100)
    threshs = kwargs.get('thresh', {8: 0.8, 16: 0.7, 32: 0.6})
    preds = {}
    targets = {}
    pos_wts = {}
    patch_sizes = sorted(patch_sizes)
    for patch_size in patch_sizes:
        shape_ = (batch_size, n_channels, im_size//patch_size, im_size//patch_size, im_size//patch_size)
        preds[str(patch_size)] = torch.rand(shape_)
        target = torch.rand(shape_).gt(threshs[patch_size]).float()
        targets[str(patch_size)] = target
        pos_wts[str(patch_size)] = [((target[:, 0, :, :, :] == 0).sum() / target[:, 0, :, :, :].sum()).item()]
    return preds, targets, pos_wts

In [2]:
def create_patch_classify_preds_and_target_multilabel(**kwargs):
    patch_sizes = kwargs.get('patch_sizes', [8, 16, 32])
    im_size = kwargs.get('im_size', 128)
    n_channels = 3
    batch_size = kwargs.get('batch_size', 100)
    threshs = kwargs.get('thresh', {8: 0.8, 16: 0.7, 32: 0.6})
    preds = {}
    targets = {}
    pos_wts = {}
    patch_sizes = sorted(patch_sizes)
    for patch_size in patch_sizes:
        shape_ = (batch_size, n_channels, im_size//patch_size, im_size//patch_size, im_size//patch_size)
        preds[str(patch_size)] = torch.rand(shape_)
        thresh = [threshs[patch_size] + i*0.03 for i in range(n_channels)]
        target = torch.rand(shape_).gt(torch.tensor(thresh).view(3, 1, 1, 1)).float()
        targets[str(patch_size)] = target
        pos_wts[str(patch_size)] = [((target[:, i, :, :, :] == 0).sum() / target[:, i, :, :, :].sum()).item() for i in range(n_channels)]
    return preds, targets, pos_wts

In [ ]:

preds, targets, pos_wts = create_patch_classify_preds_and_target_binary(batch_size=1)
print('preds', {str(k):v.shape for k,v in preds.items()})
print('targets', {str(k):v.shape for k,v in targets.items()})
print('pos_wt', pos_wts)
print('\n')

loss_fn = PatchClassifyBCELoss(mode='binary', pos_weights=pos_wts, patch_res=[8, 16, 32], scale_loss=1)
loss = loss_fn(preds, targets)
pprint(loss, sort_dicts=False)

preds {'8': torch.Size([1, 1, 16, 16, 16]), '16': torch.Size([1, 1, 8, 8, 8]), '32': torch.Size([1, 1, 4, 4, 4])}
targets {'8': torch.Size([1, 1, 16, 16, 16]), '16': torch.Size([1, 1, 8, 8, 8]), '32': torch.Size([1, 1, 4, 4, 4])}
pos_wt {'8': [4.204574108123779], '16': [2.6056337356567383], '32': [1.9090908765792847]}


{'binary_patch_classify_bce_loss': tensor(3.2363),
 'binary_patch_classify_bce_loss_WT_res=8': tensor(1.1822),
 'binary_patch_classify_bce_loss_res=8': tensor(1.1822),
 'binary_patch_classify_bce_loss_WT_res=16': tensor(1.0606),
 'binary_patch_classify_bce_loss_res=16': tensor(1.0606),
 'binary_patch_classify_bce_loss_WT_res=32': tensor(0.9935),
 'binary_patch_classify_bce_loss_res=32': tensor(0.9935)}


In [ ]:

preds, targets, pos_wts = create_patch_classify_preds_and_target_binary(batch_size=1)
print('preds', {str(k):v.shape for k,v in preds.items()})
print('targets', {str(k):v.shape for k,v in targets.items()})
print('pos_wt', pos_wts)
print('\n')

loss_fn = PatchClassifyBCELoss(mode='binary', pos_weights=None, patch_res=[8, 16, 32], scale_loss=1)
loss = loss_fn(preds, targets)
pprint(loss, sort_dicts=False)

preds {'8': torch.Size([1, 1, 16, 16, 16]), '16': torch.Size([1, 1, 8, 8, 8]), '32': torch.Size([1, 1, 4, 4, 4])}
targets {'8': torch.Size([1, 1, 16, 16, 16]), '16': torch.Size([1, 1, 8, 8, 8]), '32': torch.Size([1, 1, 4, 4, 4])}
pos_wt {'8': [3.911271095275879], '16': [2.7101449966430664], '32': [1.3703703880310059]}


{'binary_patch_classify_bce_loss': tensor(2.4971),
 'binary_patch_classify_bce_loss_WT_res=8': tensor(0.8855),
 'binary_patch_classify_bce_loss_res=8': tensor(0.8855),
 'binary_patch_classify_bce_loss_WT_res=16': tensor(0.8458),
 'binary_patch_classify_bce_loss_res=16': tensor(0.8458),
 'binary_patch_classify_bce_loss_WT_res=32': tensor(0.7658),
 'binary_patch_classify_bce_loss_res=32': tensor(0.7658)}


In [25]:

preds, targets, pos_wts = create_patch_classify_preds_and_target_multilabel(batch_size=10)
print('preds', {str(k):v.shape for k,v in preds.items()})
print('targets', {str(k):v.shape for k,v in targets.items()})
print('pos_wt', pos_wts)
print('\n')

loss_fn = PatchClassifyBCELoss(mode='multilabel', labels=['ED', 'ET', 'NCR'] , pos_weights=pos_wts, patch_res=[8, 16, 32], scale_loss=1)
loss = loss_fn(preds, targets)
pprint(loss, sort_dicts=False)

preds {'8': torch.Size([10, 3, 16, 16, 16]), '16': torch.Size([10, 3, 8, 8, 8]), '32': torch.Size([10, 3, 4, 4, 4])}
targets {'8': torch.Size([10, 3, 16, 16, 16]), '16': torch.Size([10, 3, 8, 8, 8]), '32': torch.Size([10, 3, 4, 4, 4])}
pos_wt {'8': [3.9083282947540283, 4.876614093780518, 6.132160663604736], '16': [2.250793695449829, 2.6781609058380127, 3.0667195320129395], '32': [1.5098038911819458, 1.5498007535934448, 1.9090908765792847]}


{'multilabel_patch_classify_bce_loss': tensor(3.1908),
 'multilabel_patch_classify_bce_loss_ED_res=8': tensor(1.1690),
 'multilabel_patch_classify_bce_loss_ET_res=8': tensor(1.2158),
 'multilabel_patch_classify_bce_loss_NCR_res=8': tensor(1.2629),
 'multilabel_patch_classify_bce_loss_res=8': tensor(1.2159),
 'multilabel_patch_classify_bce_loss_ED_res=16': tensor(1.0180),
 'multilabel_patch_classify_bce_loss_ET_res=16': tensor(1.0684),
 'multilabel_patch_classify_bce_loss_NCR_res=16': tensor(1.1058),
 'multilabel_patch_classify_bce_loss_res=16': ten

In [26]:
preds, targets, pos_wts = create_patch_classify_preds_and_target_multilabel(batch_size=10)
print('preds', {str(k):v.shape for k,v in preds.items()})
print('targets', {str(k):v.shape for k,v in targets.items()})
print('pos_wt', pos_wts)
print('\n')

loss_fn = PatchClassifyBCELoss(mode='multilabel', labels=['ED', 'ET', 'NCR'] , pos_weights=None, patch_res=[8, 16, 32], scale_loss=1)
loss = loss_fn(preds, targets)
pprint(loss, sort_dicts=False)

preds {'8': torch.Size([10, 3, 16, 16, 16]), '16': torch.Size([10, 3, 8, 8, 8]), '32': torch.Size([10, 3, 4, 4, 4])}
targets {'8': torch.Size([10, 3, 16, 16, 16]), '16': torch.Size([10, 3, 8, 8, 8]), '32': torch.Size([10, 3, 4, 4, 4])}
pos_wt {'8': [4.028234481811523, 4.858961582183838, 6.263699054718018], '16': [2.384005308151245, 2.7101449966430664, 3.189852714538574], '32': [1.700421929359436, 1.4902724027633667, 2.1372549533843994]}


{'multilabel_patch_classify_bce_loss': tensor(2.5478),
 'multilabel_patch_classify_bce_loss_ED_res=8': tensor(0.8846),
 'multilabel_patch_classify_bce_loss_ET_res=8': tensor(0.8998),
 'multilabel_patch_classify_bce_loss_NCR_res=8': tensor(0.9144),
 'multilabel_patch_classify_bce_loss_res=8': tensor(0.8996),
 'multilabel_patch_classify_bce_loss_ED_res=16': tensor(0.8346),
 'multilabel_patch_classify_bce_loss_ET_res=16': tensor(0.8448),
 'multilabel_patch_classify_bce_loss_NCR_res=16': tensor(0.8682),
 'multilabel_patch_classify_bce_loss_res=16': tensor